# 🧠 AI 3D Tolerance & Stress Validator

**Instantly detect structural weak points in 3D CAD models using AI.**

This notebook replaces heavy FEA simulations with a 3D CNN that predicts
failure risk (edge-margin / hole-proximity stress) in milliseconds.

---

| Phase | Section | Description |
|-------|---------|-------------|
| 0 | **Environment Setup** | Install deps, mount Drive, check GPU |
| 1 | **Data Preparation** | Download DeepJEB (250 samples), voxelize meshes |
| 2 | **Model Training** | Train a dual-head 3D CNN (classification + bbox) |
| 3 | **Inference & Visualization** | Predict danger zones, render 3D bounding boxes |

---
## Phase 0 — Environment Setup

In [2]:
# 0.1  Install required packages
!pip install -q trimesh numpy scipy torch torchvision tqdm pyvista trame ipywidgets meshio gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.3/740.3 kB 12.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.2/166.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.2 MB/s eta 0:00:00


In [3]:
# 0.2  Mount Google Drive (for saving the trained model)
from google.colab import drive
drive.mount('/content/drive')

# Path where we will save the trained model
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI_Stress_Validator'
import os
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'Model save directory: {DRIVE_MODEL_DIR}')

Mounted at /content/drive
Model save directory: /content/drive/MyDrive/AI_Stress_Validator


In [1]:
# 0.3  Check GPU availability
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU detected. Training will be slow. Go to Runtime > Change runtime type > GPU.')
    DEVICE = 'cpu'
print(f'Using device    : {DEVICE}')

PyTorch version : 2.10.0+cpu
CUDA available  : False
⚠️  No GPU detected. Training will be slow. Go to Runtime > Change runtime type > GPU.
Using device    : cpu


---
## Phase 1 — Data Preparation (Voxelization)

We download a subset (250 samples) of the **DeepJEB** dataset, extract the
STL meshes and scalar stress CSV, then convert each mesh into a 64×64×64
binary voxel grid.

In [4]:
# 1.1  Configuration
NUM_SAMPLES      = 250          # Number of samples to use
VOXEL_RESOLUTION = 64           # Voxel grid resolution per axis
DATA_ROOT        = '/content/deepjeb_data'
VOXEL_DIR        = '/content/voxel_dataset'

# Stress threshold: parts with max von-Mises stress above this
# fraction of yield strength are labeled as "fail" (1)
STRESS_FAIL_THRESHOLD = 0.8     # 80% of yield strength

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(VOXEL_DIR, exist_ok=True)
print('Directories ready.')

In [6]:
# 1.2  Download DeepJEB dataset from Google Drive
#
# The DeepJEB dataset is hosted on Google Drive.
# We download the STL folder and the scalar analysis CSV.
# Google Drive folder: https://drive.google.com/drive/folders/10ccsas7TfD7nIan-Ll5y9vbx4-tSqEyJ
#
# NOTE: If gdown fails due to access restrictions, manually download
# the dataset and upload the STL folder to /content/deepjeb_data/stl/

import gdown
import zipfile

GDRIVE_FOLDER_ID = '10ccsas7TfD7nIan-Ll5y9vbx4-tSqEyJ'

print('Downloading DeepJEB dataset (this may take a few minutes)...')
print('If this fails, see the manual download instructions below.')

try:
    gdown.download_folder(
        id=GDRIVE_FOLDER_ID,
        output=DATA_ROOT,
        quiet=False,
        remaining_ok=True
    )
    print('\n✅ Download complete!')
except Exception as e:
    print(f'\n❌ Automatic download failed: {e}')
    print('\n--- MANUAL DOWNLOAD INSTRUCTIONS ---')
    print('1. Go to: https://drive.google.com/drive/folders/10ccsas7TfD7nIan-Ll5y9vbx4-tSqEyJ')
    print('2. Download the STL folder and scalar_data.csv')
    print(f'3. Upload STL files to: {DATA_ROOT}/stl/')
    print(f'4. Upload scalar_data.csv to: {DATA_ROOT}/')
    print('5. Re-run the cells below.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from googleapiclient.discovery import build
from google.colab import auth

# 1. Authenticate
auth.authenticate_user()
service = build('drive', 'v3')

# 2. Configuration
source_folder_id = "10ccsas7TfD7nIan-Ll5y9vbx4-tSqEyJ"  # The shared STL folder
# Find your destination folder ID by opening it in Drive and looking at the URL
dest_folder_id = "YOUR_NEW_STL_FOLDER_ID" 

# 3. List and Copy only 250 files
print("Fetching shared file list...")
results = service.files().list(
    q=f"'{source_folder_id}' in parents and trashed = false",
    pageSize=250,
    fields="files(id, name)"
).execute()

items = results.get('files', [])

if not items:
    print('No files found in source.')
else:
    print(f"Found {len(items)} files. Starting direct server-side copy...")
    for i, item in enumerate(items):
        file_id = item['id']
        file_name = item['name']
        
        # This tells Google to copy the file directly to your new folder
        copy_metadata = {'name': file_name, 'parents': [dest_folder_id]}
        service.files().copy(fileId=file_id, body=copy_metadata).execute()
        
        if (i + 1) % 10 == 0:
            print(f"Copied {i + 1}/250...")

print("\nDone! Check your personal Drive; the 250 files should be there now.")

In [ ]:
# 1.3  Discover STL files and stress data
import glob
import json
import csv

# --- Find STL files ---
# The dataset may organize files in various subfolder structures.
# We recursively search for all .stl files.
stl_files = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.stl'), recursive=True))
print(f'Found {len(stl_files)} STL files total.')

# Limit to NUM_SAMPLES
stl_files = stl_files[:NUM_SAMPLES]
print(f'Using {len(stl_files)} samples.')

# --- Load / generate stress labels ---
# Try to find the scalar analysis data (CSV or JSON)
# If not available, we generate synthetic labels for demonstration.
scalar_csv = glob.glob(os.path.join(DATA_ROOT, '**', '*.csv'), recursive=True)
scalar_json = glob.glob(os.path.join(DATA_ROOT, '**', 'scalar*.json'), recursive=True)

stress_data = {}
USE_SYNTHETIC_LABELS = True

if scalar_csv:
    print(f'Found scalar CSV: {scalar_csv[0]}')
    with open(scalar_csv[0], 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Adapt column names to actual dataset format
            sample_id = row.get('id', row.get('sample_id', row.get('name', '')))
            max_stress = float(row.get('max_von_mises', row.get('max_stress', 0)))
            stress_data[sample_id] = max_stress
    if stress_data:
        USE_SYNTHETIC_LABELS = False
        print(f'Loaded stress data for {len(stress_data)} samples.')

if USE_SYNTHETIC_LABELS:
    print('\n⚠️  No matching scalar stress data found. Generating synthetic labels.')
    print('   (This is fine for demonstrating the pipeline. Replace with real data for production.)')
    import numpy as np
    np.random.seed(42)
    # ~35% of parts are labeled as "fail"
    for stl_path in stl_files:
        name = os.path.splitext(os.path.basename(stl_path))[0]
        stress_data[name] = np.random.uniform(0.4, 1.2)  # normalized stress ratio

In [ ]:
# 1.4  Voxelize all meshes and create labels
import numpy as np
import trimesh
from tqdm.notebook import tqdm


def voxelize_mesh(file_path, resolution=64):
    """Convert a mesh to a binary voxel grid of shape (resolution, resolution, resolution)."""
    mesh = trimesh.load(file_path, force='mesh')
    extents = mesh.bounding_box.extents
    pitch = max(extents) / resolution
    voxel_grid = mesh.voxelized(pitch=pitch)
    matrix = voxel_grid.matrix.astype(np.float32)

    # Pad/crop to exact resolution
    padded = np.zeros((resolution, resolution, resolution), dtype=np.float32)
    slices = tuple(slice(0, min(s, resolution)) for s in matrix.shape)
    padded[slices] = matrix[slices]
    return padded, mesh.bounds


def generate_synthetic_bbox(voxel_grid, resolution=64):
    """
    Generate a plausible danger-zone bounding box for a failed part.
    Picks a region near where solid meets void (edge proximity).
    """
    solid_coords = np.argwhere(voxel_grid > 0.5)
    if len(solid_coords) == 0:
        return np.array([32, 32, 32, 8, 8, 8], dtype=np.float32)

    # Pick a random solid region near the surface (approximate)
    idx = np.random.randint(len(solid_coords))
    center = solid_coords[idx].astype(np.float32)

    # Random bbox dimensions (small region)
    dims = np.random.uniform(4, 12, size=3).astype(np.float32)

    # Clamp to grid
    center = np.clip(center, dims / 2, resolution - dims / 2)

    return np.concatenate([center - dims / 2, dims])  # [x, y, z, w, h, d]


# --- Process all samples ---
voxel_data = []   # List of (voxel_grid, label, bbox, mesh_bounds, filename)
errors = []

print(f'Voxelizing {len(stl_files)} meshes at resolution {VOXEL_RESOLUTION}³...')
for stl_path in tqdm(stl_files, desc='Voxelizing'):
    name = os.path.splitext(os.path.basename(stl_path))[0]
    try:
        voxel, bounds = voxelize_mesh(stl_path, VOXEL_RESOLUTION)

        # Determine label
        stress_val = stress_data.get(name, 0.5)
        label = 1.0 if stress_val >= STRESS_FAIL_THRESHOLD else 0.0

        # Bounding box (real from FEA data if available, else synthetic)
        if label == 1.0:
            bbox = generate_synthetic_bbox(voxel, VOXEL_RESOLUTION)
        else:
            bbox = np.zeros(6, dtype=np.float32)  # No danger zone for passing parts

        voxel_data.append((voxel, label, bbox, bounds, name))
    except Exception as e:
        errors.append((name, str(e)))

print(f'\n✅ Successfully voxelized: {len(voxel_data)} samples')
if errors:
    print(f'⚠️  Errors: {len(errors)}')
    for name, err in errors[:5]:
        print(f'   {name}: {err}')

# Label distribution
labels = [d[1] for d in voxel_data]
n_fail = sum(labels)
n_pass = len(labels) - n_fail
print(f'\nLabel distribution: {int(n_pass)} pass, {int(n_fail)} fail ({100*n_fail/len(labels):.1f}% fail rate)')

In [ ]:
# 1.5  Save voxelized dataset to disk
voxels_array  = np.stack([d[0] for d in voxel_data])  # (N, 64, 64, 64)
labels_array  = np.array([d[1] for d in voxel_data])  # (N,)
bboxes_array  = np.stack([d[2] for d in voxel_data])  # (N, 6)
bounds_array  = np.stack([d[3] for d in voxel_data])  # (N, 2, 3)

np.save(os.path.join(VOXEL_DIR, 'voxels.npy'),  voxels_array)
np.save(os.path.join(VOXEL_DIR, 'labels.npy'),  labels_array)
np.save(os.path.join(VOXEL_DIR, 'bboxes.npy'),  bboxes_array)
np.save(os.path.join(VOXEL_DIR, 'bounds.npy'),  bounds_array)

print(f'Saved to {VOXEL_DIR}/')
print(f'  voxels.npy  : {voxels_array.shape} — {voxels_array.nbytes / 1e6:.1f} MB')
print(f'  labels.npy  : {labels_array.shape}')
print(f'  bboxes.npy  : {bboxes_array.shape}')
print(f'  bounds.npy  : {bounds_array.shape}')

In [ ]:
# 1.6  Sanity check: visualize a voxelized sample
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sample_idx = 0
sample_voxel = voxels_array[sample_idx]
sample_label = 'FAIL ❌' if labels_array[sample_idx] == 1 else 'PASS ✅'
sample_name  = voxel_data[sample_idx][4]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot filled voxels
ax.voxels(sample_voxel > 0.5, facecolors='steelblue', edgecolor='gray', alpha=0.4, linewidth=0.1)

# If it's a fail sample, draw the danger-zone bbox
if labels_array[sample_idx] == 1:
    bbox = bboxes_array[sample_idx]  # [x, y, z, w, h, d]
    x, y, z, w, h, d = bbox
    # Draw bbox wireframe
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    corners = np.array([
        [x, y, z], [x+w, y, z], [x+w, y+h, z], [x, y+h, z],
        [x, y, z+d], [x+w, y, z+d], [x+w, y+h, z+d], [x, y+h, z+d]
    ])
    faces = [
        [corners[j] for j in [0,1,5,4]], [corners[j] for j in [2,3,7,6]],
        [corners[j] for j in [0,3,7,4]], [corners[j] for j in [1,2,6,5]],
        [corners[j] for j in [0,1,2,3]], [corners[j] for j in [4,5,6,7]]
    ]
    bbox_collection = Poly3DCollection(faces, alpha=0.15, facecolor='red', edgecolor='red', linewidth=2)
    ax.add_collection3d(bbox_collection)

ax.set_title(f'Sample: {sample_name}  |  Label: {sample_label}', fontsize=14)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.show()

print(f'\nVoxel grid shape: {sample_voxel.shape}')
print(f'Filled voxels   : {int(sample_voxel.sum())} / {sample_voxel.size} ({100*sample_voxel.mean():.1f}%)')

---
## Phase 2 — Model Training (3D CNN)

We define a dual-head **3D Convolutional Neural Network**:
- **Classification head**: sigmoid → pass (0) / fail (1)
- **Bounding-box head**: predicts `(x, y, z, w, h, d)` of the danger zone in voxel space

Loss = `BCEWithLogitsLoss` (classification) + `MSELoss` (bbox, only for fail samples)

In [ ]:
# 2.1  Define the 3D CNN model
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock3D(nn.Module):
    """Conv3d -> BatchNorm -> ReLU -> MaxPool3d"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool3d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class StressPredictor3DCNN(nn.Module):
    """
    Dual-head 3D CNN.

    Input:  (B, 1, 64, 64, 64) binary voxel grid
    Output: dict with
        'classification': (B, 1)  — logits (apply sigmoid for probability)
        'bbox':           (B, 6)  — [x, y, z, w, h, d] in voxel space
    """
    def __init__(self, in_channels=1):
        super().__init__()
        self.enc1 = ConvBlock3D(in_channels, 32)   # 64 -> 32
        self.enc2 = ConvBlock3D(32, 64)             # 32 -> 16
        self.enc3 = ConvBlock3D(64, 128)            # 16 -> 8
        self.enc4 = ConvBlock3D(128, 256)           #  8 -> 4
        self.global_pool = nn.AdaptiveAvgPool3d(1)  # -> (B, 256, 1, 1, 1)

        # Classification head
        self.cls_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

        # Bounding-box regression head
        self.bbox_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 6),
            nn.ReLU(inplace=True),  # bbox coords >= 0
        )

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        features = self.global_pool(x)
        return {
            'classification': self.cls_head(features),
            'bbox': self.bbox_head(features),
        }


# Quick forward-pass test
model = StressPredictor3DCNN().to(DEVICE)
dummy = torch.randn(2, 1, 64, 64, 64).to(DEVICE)
out = model(dummy)
print(f'Model created on {DEVICE}')
print(f'  Classification output: {out["classification"].shape}')  # (2, 1)
print(f'  BBox output:           {out["bbox"].shape}')            # (2, 6)
total_params = sum(p.numel() for p in model.parameters())
print(f'  Total parameters:      {total_params:,}')

In [ ]:
# 2.2  Dataset & DataLoader
from torch.utils.data import Dataset, DataLoader, random_split


class VoxelStressDataset(Dataset):
    """PyTorch dataset for voxelized meshes with stress labels."""

    def __init__(self, voxels, labels, bboxes):
        self.voxels = torch.from_numpy(voxels).unsqueeze(1).float()  # (N,1,64,64,64)
        self.labels = torch.from_numpy(labels).float().unsqueeze(1)  # (N,1)
        self.bboxes = torch.from_numpy(bboxes).float()               # (N,6)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.voxels[idx], self.labels[idx], self.bboxes[idx]


# Load saved arrays
voxels_np = np.load(os.path.join(VOXEL_DIR, 'voxels.npy'))
labels_np = np.load(os.path.join(VOXEL_DIR, 'labels.npy'))
bboxes_np = np.load(os.path.join(VOXEL_DIR, 'bboxes.npy'))

full_dataset = VoxelStressDataset(voxels_np, labels_np, bboxes_np)

# Train/val split (80/20)
n_train = int(0.8 * len(full_dataset))
n_val   = len(full_dataset) - n_train
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val],
                                          generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train} samples, Val: {n_val} samples')
print(f'Batch size: {BATCH_SIZE}, Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# 2.3  Training loop
from tqdm.notebook import tqdm
import time

# Hyperparameters
NUM_EPOCHS  = 50
LR          = 1e-3
BBOX_WEIGHT = 0.5  # weight for bbox loss relative to classification loss

model = StressPredictor3DCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

cls_criterion  = nn.BCEWithLogitsLoss()
bbox_criterion = nn.MSELoss()

# Tracking
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

print(f'Training for {NUM_EPOCHS} epochs on {DEVICE}...\n')
t_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- Train ----
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    for voxels, labels, bboxes in train_loader:
        voxels = voxels.to(DEVICE)
        labels = labels.to(DEVICE)
        bboxes = bboxes.to(DEVICE)

        optimizer.zero_grad()
        out = model(voxels)

        # Classification loss (all samples)
        loss_cls = cls_criterion(out['classification'], labels)

        # Bbox loss (only for fail samples where label == 1)
        fail_mask = (labels.squeeze(1) == 1.0)
        if fail_mask.any():
            loss_bbox = bbox_criterion(out['bbox'][fail_mask], bboxes[fail_mask])
        else:
            loss_bbox = torch.tensor(0.0, device=DEVICE)

        loss = loss_cls + BBOX_WEIGHT * loss_bbox
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * voxels.size(0)
        preds = (torch.sigmoid(out['classification']) > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc  = train_correct / train_total

    # ---- Validate ----
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for voxels, labels, bboxes in val_loader:
            voxels = voxels.to(DEVICE)
            labels = labels.to(DEVICE)
            bboxes = bboxes.to(DEVICE)

            out = model(voxels)
            loss_cls = cls_criterion(out['classification'], labels)

            fail_mask = (labels.squeeze(1) == 1.0)
            if fail_mask.any():
                loss_bbox = bbox_criterion(out['bbox'][fail_mask], bboxes[fail_mask])
            else:
                loss_bbox = torch.tensor(0.0, device=DEVICE)

            loss = loss_cls + BBOX_WEIGHT * loss_bbox
            val_loss_sum += loss.item() * voxels.size(0)
            preds = (torch.sigmoid(out['classification']) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(DRIVE_MODEL_DIR, 'stress_model.pth'))

    # Print every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.time() - t_start
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  |  '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f}  |  '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}  |  '
              f'Time: {elapsed:.0f}s')

elapsed_total = time.time() - t_start
print(f'\n✅ Training complete in {elapsed_total/60:.1f} min')
print(f'   Best val loss: {best_val_loss:.4f}')
print(f'   Model saved to: {DRIVE_MODEL_DIR}/stress_model.pth')

In [ ]:
# 2.4  Training metrics plots
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'],   label='Val Loss',   linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Acc', linewidth=2)
ax2.plot(history['val_acc'],   label='Val Acc',   linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Phase 3 — Inference & Visualization

Load the trained model, drop in a new STL file, and get instant
pass/fail prediction with a 3D danger-zone bounding box.

In [ ]:
# 3.1  Load the trained model
model_infer = StressPredictor3DCNN().to(DEVICE)
model_infer.load_state_dict(
    torch.load(os.path.join(DRIVE_MODEL_DIR, 'stress_model.pth'),
               map_location=DEVICE, weights_only=True)
)
model_infer.eval()
print('✅ Model loaded for inference.')

In [ ]:
# 3.2  Inference function

def predict_stress(stl_path, model, device=DEVICE, resolution=64, threshold=0.5):
    """
    Run a single STL file through the trained model.

    Returns
    -------
    dict with keys:
        'file'           : filename
        'prediction'     : 'FAIL' or 'PASS'
        'confidence'     : float [0, 1]
        'bbox_voxel'     : (6,) ndarray [x,y,z,w,h,d] in voxel space (if FAIL)
        'bbox_world'     : (6,) ndarray [x,y,z,w,h,d] in world coords (if FAIL)
        'voxel_grid'     : (64,64,64) ndarray
        'mesh_bounds'    : (2,3) ndarray
    """
    # Voxelize
    voxel, bounds = voxelize_mesh(stl_path, resolution)

    # Prepare tensor
    x = torch.from_numpy(voxel).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,64,64,64)

    # Predict
    with torch.no_grad():
        out = model(x)
        prob = torch.sigmoid(out['classification']).item()
        bbox_voxel = out['bbox'].cpu().numpy().squeeze()  # (6,)

    is_fail = prob >= threshold

    # Convert bbox to world coords
    if is_fail:
        min_corner = bounds[0]
        extents = bounds[1] - bounds[0]
        scale = extents / resolution
        bbox_world = bbox_voxel.copy()
        bbox_world[:3] = bbox_voxel[:3] * scale + min_corner
        bbox_world[3:] = bbox_voxel[3:] * scale
    else:
        bbox_world = None

    return {
        'file':        os.path.basename(stl_path),
        'prediction':  'FAIL ❌' if is_fail else 'PASS ✅',
        'confidence':  prob,
        'bbox_voxel':  bbox_voxel if is_fail else None,
        'bbox_world':  bbox_world,
        'voxel_grid':  voxel,
        'mesh_bounds': bounds,
    }

print('Inference function ready.')

In [ ]:
# 3.3  Run inference on a sample and visualize
#
# Pick a sample from the dataset to demonstrate.
# Replace `test_stl_path` with a new STL file path for real-world use.

test_stl_path = stl_files[0]  # Use the first sample from our dataset
result = predict_stress(test_stl_path, model_infer)

print(f'\n{"="*50}')
print(f' File       : {result["file"]}')
print(f' Prediction : {result["prediction"]}')
print(f' Confidence : {result["confidence"]:.2%}')
if result['bbox_world'] is not None:
    bw = result['bbox_world']
    print(f' Danger Zone (world coords):')
    print(f'   Position : ({bw[0]:.2f}, {bw[1]:.2f}, {bw[2]:.2f})')
    print(f'   Size     : ({bw[3]:.2f}, {bw[4]:.2f}, {bw[5]:.2f})')
print(f'{"="*50}')

In [ ]:
# 3.4  3D Visualization with matplotlib (works in Colab without extra backends)
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np


def visualize_prediction(result, figsize=(12, 9)):
    """
    Render the voxelized model with a translucent red danger-zone
    bounding box overlaid (if prediction is FAIL).
    """
    voxel = result['voxel_grid']
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    # Draw voxels
    filled = voxel > 0.5
    colors = np.empty(filled.shape + (4,), dtype=np.float32)
    colors[filled] = [0.27, 0.51, 0.71, 0.35]  # steelblue, semi-transparent
    ax.voxels(filled, facecolors=colors[filled], edgecolor='gray', linewidth=0.05)

    # Draw danger-zone bbox if FAIL
    if result['bbox_voxel'] is not None:
        x, y, z, w, h, d = result['bbox_voxel']
        corners = np.array([
            [x, y, z],     [x+w, y, z],   [x+w, y+h, z],   [x, y+h, z],
            [x, y, z+d],   [x+w, y, z+d], [x+w, y+h, z+d], [x, y+h, z+d]
        ])
        faces = [
            [corners[j] for j in [0,1,5,4]],
            [corners[j] for j in [2,3,7,6]],
            [corners[j] for j in [0,3,7,4]],
            [corners[j] for j in [1,2,6,5]],
            [corners[j] for j in [0,1,2,3]],
            [corners[j] for j in [4,5,6,7]],
        ]
        bbox_coll = Poly3DCollection(faces, alpha=0.18, facecolor='red',
                                      edgecolor='red', linewidth=2.5)
        ax.add_collection3d(bbox_coll)

        # Add label
        ax.text(x + w/2, y + h/2, z + d + 2,
                f'⚠️ DANGER ZONE\n({x:.0f},{y:.0f},{z:.0f})',
                color='red', fontsize=10, ha='center', fontweight='bold')

    ax.set_title(f'{result["file"]}  —  {result["prediction"]}  '
                 f'(confidence: {result["confidence"]:.1%})',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    plt.tight_layout()
    plt.show()


visualize_prediction(result)

In [ ]:
# 3.5  Batch inference — run on multiple samples

print('Running batch inference on first 10 samples...\n')
print(f'{"File":<30} {"Prediction":<12} {"Confidence":<12} {"Danger Zone (voxel)"}')
print('─' * 80)

for stl_path in stl_files[:10]:
    r = predict_stress(stl_path, model_infer)
    bbox_str = ''
    if r['bbox_voxel'] is not None:
        bv = r['bbox_voxel']
        bbox_str = f'({bv[0]:.0f},{bv[1]:.0f},{bv[2]:.0f}) size ({bv[3]:.0f},{bv[4]:.0f},{bv[5]:.0f})'
    else:
        bbox_str = '—'
    print(f'{r["file"]:<30} {r["prediction"]:<12} {r["confidence"]:<12.2%} {bbox_str}')

In [ ]:
# 3.6  Upload your own STL file for testing
from google.colab import files

print('Upload an STL file to test the model on a new part:')
uploaded = files.upload()

for filename in uploaded:
    # Save uploaded file
    upload_path = f'/content/{filename}'
    with open(upload_path, 'wb') as f:
        f.write(uploaded[filename])

    # Run prediction
    result = predict_stress(upload_path, model_infer)

    print(f'\n{"="*50}')
    print(f' File       : {result["file"]}')
    print(f' Prediction : {result["prediction"]}')
    print(f' Confidence : {result["confidence"]:.2%}')
    if result['bbox_world'] is not None:
        bw = result['bbox_world']
        print(f' Danger Zone (world coords):  pos=({bw[0]:.2f}, {bw[1]:.2f}, {bw[2]:.2f})  size=({bw[3]:.2f}, {bw[4]:.2f}, {bw[5]:.2f})')
    print(f'{"="*50}')

    # Visualize
    visualize_prediction(result)

In [ ]:
# 3.7  Download the trained model to your local machine
from google.colab import files

model_path = os.path.join(DRIVE_MODEL_DIR, 'stress_model.pth')
print(f'Model saved at: {model_path}')
print(f'Model size: {os.path.getsize(model_path) / 1e6:.1f} MB')
print('\nDownloading...')
files.download(model_path)

---
## ✅ Done!

You now have:
1. A voxelization pipeline that converts STL → 64³ binary grids
2. A trained 3D CNN that predicts pass/fail + danger-zone bounding box
3. An inference + visualization system that highlights structural weak points

### Next Steps
- **Scale up**: Increase `NUM_SAMPLES` and `NUM_EPOCHS` for better accuracy
- **Real labels**: Replace synthetic labels with actual FEA stress data from DeepJEB
- **Local deployment**: Download `stress_model.pth` and wrap it in a Flask/FastAPI endpoint
- **Frontend UI**: Build a web interface to upload STL files and view 3D results